# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load, inspect, and process the FAIR^2 dataset using the `mlcroissant` library. You will learn how to work directly with Croissant metadata, access fields and records, and perform exploratory data analysis using clear references by `@id`.

### Dataset Source
The dataset is defined by a Croissant schema:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# View basic metadata
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review the available record sets, fields, and their `@id`s.

We list all `recordSet` objects by their `@id` and show their key fields. Fields and columns are always referenced by `@id`, following FAIR and Croissant best practices.

In [ ]:
# List all record sets in the dataset
record_sets = list(dataset.record_sets())

print('Available Record Sets:')
for rs in record_sets:
    print(f"- @id: {rs['@id']}")
    rs_fields = rs.get('field', [])
    if isinstance(rs_fields, dict):  # single field
        rs_fields = [rs_fields]
    print('  Fields:')
    for field in rs_fields:
        if isinstance(field, dict):
            print(f"    - @id: {field.get('@id', '<no-id>')} | name: {field.get('name', '<no-name>')}")


## 3. Data Extraction
Load records from a specific record set into a DataFrame. Use only record set and field `@id`s from the previous overview.

For this dataset, we select the main tabular record set, typically named with a descriptive `@id` (`/CancerCases`, for example). You may adjust the `record_set_id` below based on the output above.

In [ ]:
# Select the primary record set by its @id
# (adjust to match the main table, based on the output above; here we use an example @id)
# To list all records for all record sets, iterate through them programmatically

# Extract all records for all tabular record sets
dataframes = {}
main_record_set_id = None

for rs in record_sets:
    rs_id = rs['@id']
    print(f"\nLoading data for record set: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        if main_record_set_id is None or df.shape[0] > dataframes[main_record_set_id].shape[0]:
            main_record_set_id = rs_id
        print(f"  Loaded {df.shape[0]} records, columns: {df.columns.tolist()}")
    else:
        print("  No records available.")

if main_record_set_id:
    print(f"\nUsing record set '{main_record_set_id}' as the primary table for EDA.")
    display(dataframes[main_record_set_id].head())
else:
    print("No main tabular record set found. Check record set @id.")

## 4. Exploratory Data Analysis (EDA)
Let's examine, filter, and process some of the numeric and categorical fields in the dataset. 

You should always refer to fields using their full `@id`. Numeric fields might include age at diagnosis or diagnosis interval in months. We use these for filtering and normalization below.

In [ ]:
# Find candidate numeric and group fields from the available DataFrame
df = dataframes[main_record_set_id]
print('Sample columns:', df.columns.tolist())

# For demonstration, suppose the numeric field is '@id': '/interval_months', and the group by field is '@id': '/sex'
# Update these @ids as needed based on your record set fields
numeric_field_id = None
group_field_id = None

# Try to auto-select likely fields by @id or column name patterns
candidates = [c for c in df.columns if ('interval' in c or 'age' in c or 'months' in c) and df[c].dtype in ['int64', 'float64']]
if candidates:
    numeric_field_id = candidates[0]
    print(f"Using numeric field for EDA: {numeric_field_id}")
else:
    print('No candidate numeric field found.')

group_candidates = [c for c in df.columns if 'sex' in c.lower() or 'msi' in c.lower() or 'site' in c.lower() or 'location' in c.lower()]
if group_candidates:
    group_field_id = group_candidates[0]
    print(f"Using group-by field: {group_field_id}")
else:
    print('No candidate group-by field found.')

# Only run EDA if a numeric field is present
if numeric_field_id:
    # Filter records with the numeric field above a threshold (10 as example)
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # Group by a candidate categorical field if available
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].agg(['mean', 'count', 'std'])
        print(f"Grouped {numeric_field_id} by {group_field_id}:")
        display(grouped_df)
else:
    print('No numeric field available for EDA.')

## 5. Visualization
Visualize the distribution of the primary numeric variable and its relationship with the selected group field (if any).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only plot if a numeric field is available
if numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id], kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.show()

## 6. Conclusion

In this notebook, we've demonstrated how to use the `mlcroissant` library to load a Croissant-defined dataset, explore its structure via `@id` references, extract tabular data, and perform introductory analysis including filtering, normalization, grouping, and visualization—all while maintaining strict references by `@id` per FAIR best practices. This workflow provides an interoperable, reproducible foundation for researching clinicopathological predictors in second primary colorectal cancer among cancer survivors.